# 02 — Routing Analysis

MoE routing behavior: expert-load balance and entropy over training (from the run log),
and the modality × expert specialization heatmap. This is the core MoE analysis of the
project. Reads `results/real_run_metrics_300steps.jsonl` and, if a checkpoint is present,
recomputes the heatmap; otherwise it shows the committed figure.

In [ ]:
import sys, json
from pathlib import Path
import matplotlib.pyplot as plt
REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO))
rows = [json.loads(l) for l in open(REPO / 'results' / 'real_run_metrics_300steps.jsonl') if 'routing/load_balance' in l]
steps = [r['step'] for r in rows]
print(f'{len(rows)} logged training steps')

In [ ]:
# Load balance (load-entropy / max) and gate entropy over training
fig, ax = plt.subplots(1, 2, figsize=(13,4))
ax[0].plot(steps, [r['routing/load_balance'] for r in rows], marker='o', ms=3)
ax[0].axhline(1.0, ls='--', c='gray', label='perfect balance')
ax[0].set_title('Expert load balance (↑ better)'); ax[0].set_xlabel('step'); ax[0].set_ylim(0.8,1.02); ax[0].legend()
ax[1].plot(steps, [r['routing/gate_entropy'] for r in rows], marker='o', ms=3, label='gate entropy')
ax[1].plot(steps, [r['routing/load_entropy'] for r in rows], marker='o', ms=3, label='load entropy')
ax[1].axhline(2.079, ls='--', c='gray', label='max = log 8')
ax[1].set_title('Routing entropy (nats)'); ax[1].set_xlabel('step'); ax[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# Per-expert load at the final logged step
last = rows[-1]
experts = sorted((k for k in last if k.startswith('routing/load/expert_')), key=lambda k: int(k.rsplit('_',1)[1]))
loads = [last[k] for k in experts]
plt.figure(figsize=(8,3))
plt.bar(range(len(loads)), loads, color='tab:purple')
plt.axhline(1/len(loads), ls='--', c='gray', label='uniform (1/8)')
plt.title(f"Per-expert load @ step {last['step']}"); plt.xlabel('expert'); plt.ylabel('token fraction'); plt.legend(); plt.show()

In [ ]:
# Modality × expert specialization heatmap.
# If a trained checkpoint exists, recompute; otherwise show the committed figure.
from IPython.display import Image, display
ckpt = REPO / 'checkpoints' / 'small-moe-baseline' / 'final'
if (ckpt / 'hf').exists():
    from src.data.tokenizer import MODALITIES, build_tokenizer
    from src.data.sources import iter_local_examples, take
    from src.eval.routing_analysis import analyze_routing, save_heatmap
    from src.model.model import SmallMoE
    tok = build_tokenizer(); model = SmallMoE.from_pretrained(ckpt)
    ex = {m: take(iter_local_examples(REPO/'data'/'raw'/f'{m}.jsonl'), 40) for m in MODALITIES}
    a = analyze_routing(model, tok, ex, max_len=256, max_examples=40)
    print('specialization score:', round(a.specialization_score(),3))
    save_heatmap(a, REPO/'report'/'figures'/'routing_heatmap.png')
display(Image(str(REPO/'report'/'figures'/'routing_heatmap.png')))

**Reading the heatmap.** Brighter cells = a modality routing a larger token fraction to
that expert. The observed pattern is *diffuse* specialization: some experts lean toward a
modality (e.g. one toward math, another toward logic), but experts are shared across
modalities and each modality spreads over several experts. This refutes the naive "one
expert per modality" intuition and matches the OLMoE findings — motivating 8 experts with
top-2 routing rather than 4 experts for 4 modalities.